In [21]:
import pandas as pd

In [ ]:
df_clean = pd.read_excel('us_state_ai_governance_legislation_tracker_cleaned.xlsx', header=0, sheet_name=0)
df_url = pd.read_excel('us_state_ai_governance_legislation_tracker_cleaned.xlsx', header=0, sheet_name=1)

In [ ]:
df_clean['Jurisdiction'] = df_clean['Jurisdiction'].ffill()

In [28]:
# Remove redundant spaces from column names
df_clean.columns = df_clean.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Display cleaned column names
print("Cleaned column names:")
print(df_clean.columns.tolist())


Cleaned column names:
['Jurisdiction', 'Legislative process', 'Introduced', 'In committee', 'In cross chamber', 'In cross committee', 'Passed', 'Signed', 'Statute/bill', 'Scope', 'Governance', 'Program and documentation', 'Assessments', 'Training', 'Responsible individual', 'Transparency', 'General notice', 'Labeling/notification', 'Explanation/incident reporting', 'Developer documentation', 'Assurance', 'Registration', 'Third-party review', 'Individual rights', 'Opt out/appeal', 'Nondiscrimination']


In [ ]:
# Extract the hyperlink from the 'Statute/bill_url' column in df_url and add it to df_clean as a new column 'url'

import re
import openpyxl

# Read the Excel file directly with openpyxl to preserve hyperlinks
wb = openpyxl.load_workbook('us_state_ai_governance_legislation_tracker_cleaned.xlsx', data_only=False)
ws = wb.worksheets[1]  # Second sheet (index 1)

# Find the column index for 'Statute/bill_url'
header_row = 1
statute_col_idx = None
for col in range(1, ws.max_column + 1):
    if ws.cell(row=header_row, column=col).value and 'Statute/bill' in str(ws.cell(row=header_row, column=col).value):
        statute_col_idx = col
        break

# Extract hyperlinks
urls = []
if statute_col_idx:
    for row in range(2, len(df_url) + 2):  # Start from row 2 (after header)
        cell = ws.cell(row=row, column=statute_col_idx)
        if cell.hyperlink:
            urls.append(cell.hyperlink.target)
        elif cell.value and isinstance(cell.value, str) and cell.value.startswith('http'):
            urls.append(cell.value)
        else:
            urls.append(None)
else:
    print("Warning: 'Statute/bill_url' column not found")
    urls = [None] * len(df_url)

# Add the URLs to df_clean
df_clean['url'] = urls[:len(df_clean)]

df_clean['url']

Extracted 56 URLs from 56 rows


c:\Users\meier\OneDrive\Documents\ai_policy\.venv\Lib\site-packages\openpyxl\reader\drawings.py:29: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")


In [ ]:
# Remove excessive spaces from the 'url' column in df_clean
df_clean['url'] = df_clean['url'].astype(str).str.replace(r'\s+', '', regex=True).replace({'None': None})
df_clean['url']

0     https://leginfo.legislature.ca.gov/faces/billT...
1     https://leginfo.legislature.ca.gov/faces/billT...
2               https://leg.colorado.gov/bills/sb24-205
3     https://le.utah.gov/~2024/bills/static/SB0149....
4     https://le.utah.gov/~2025/bills/static/SB0226....
5     https://capitol.texas.gov/BillLookup/History.a...
6     https://leginfo.legislature.ca.gov/faces/billN...
7     https://leginfo.legislature.ca.gov/faces/billS...
8     https://leginfo.legislature.ca.gov/faces/billC...
9     https://leginfo.legislature.ca.gov/faces/billT...
10    https://leginfo.legislature.ca.gov/faces/billC...
11    https://www.capitol.hawaii.gov/session/measure...
12    https://www.ilga.gov/legislation/BillStatus.as...
13    https://www.ilga.gov/legislation/BillStatus.as...
14    https://www.ilga.gov/legislation/BillStatus.as...
15    https://www.ilga.gov/legislation/BillStatus.as...
16    https://www.legis.iowa.gov/legislation/BillBoo...
17            https://malegislature.gov/Bills/19

In [36]:
# Save to CSV with UTF-8 encoding to handle special characters
df_clean.to_csv('us_state_ai_governance_legislation_tracker_processed.csv', 
                index=False, 
                encoding='utf-8-sig')  # utf-8-sig adds BOM for Excel compatibility
print("Saved to: us_state_ai_governance_legislation_tracker_processed.csv")
print("Location: Current directory (C:\\Users\\meier\\OneDrive\\Documents\\ai_policy)")


Saved to: us_state_ai_governance_legislation_tracker_processed.csv
Location: Current directory (C:\Users\meier\OneDrive\Documents\ai_policy)
